# Chapter 5 — Similarity Learning: Two-Tower Retrieval Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github//modern-recommender-systems/blob/main/notebooks/chapter-05/01_similarity_learning.ipynb)


This notebook demonstrates the similarity-learning components of Sections 5.1
and 5.3. The implementations live in the `recsys` package.

The notebook itself only loads data, runs the training, and inspects the
results — the numbers behind Table 5.X. It saves the trained embeddings to
`data/processed/chapter05/`, which `02_ann_retrieval.ipynb` and
`02_full_pipeline.ipynb` load. Run this notebook first.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

In [ ]:
import json
import random

import mlflow
import numpy as np
import pandas as pd
import torch

from recsys.data.loaders import load_movielens
from recsys.data.preprocessing import (
    add_item_idx, build_item_index, filter_min_item_ratings, filter_positive,
    sample_active_users, temporal_split_per_user, user_item_lists,
)
from recsys.fourstage_recsys.retrieval.two_tower import (
    TwoTower, TwoTowerWithInfoNCE, create_negative_pairs,
    create_positive_pairs, extract_embeddings, train_bce, train_infonce,
)
from recsys.evaluation.retrieval import evaluate_embedding_space

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
mlflow.set_experiment("chapter-05-two-tower")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

## 1. Data: a sample of MovieLens 25M

As everywhere in this book, we work with MovieLens 25M, sampled down to a set
of active users so the models train in minutes on a laptop — the same protocol
as Chapter 4. Ratings of 4.0 and above count as positive interactions, the
most recent 20% of each user's interactions are held out for testing, and the
item index mappings are built exactly once (see the ID contract in
`recsys.data.preprocessing`: pipeline stages speak string item IDs, models
speak dense indices, and these mappings are the only bridge).

In [ ]:
ratings, movies = load_movielens("ml-25m", data_dir=project_root / "data")

ratings = sample_active_users(ratings, n_users=10_000, min_ratings=20, seed=SEED)
ratings = filter_min_item_ratings(ratings, min_ratings=10)
interactions = filter_positive(ratings, threshold=4.0)

item_ids, item_to_idx, idx_to_title, idx_to_genres = build_item_index(
    interactions, movies)
interactions = add_item_idx(interactions, item_to_idx)
num_items = len(item_ids)

train_df, test_df = temporal_split_per_user(interactions, test_frac=0.2)
train_items_idx = user_item_lists(train_df, item_col="item_idx")
relevance_sets = test_df.groupby("userId")["item_idx"].apply(set).to_dict()

print(f"{interactions.userId.nunique():,} users, {num_items:,} movies, "
      f"{len(train_df):,} train / {len(test_df):,} test interactions")

## 2. Positive and negative pairs (Listings 5.1 and 5.2)

A positive pair is two items the same user engaged with; a negative pair
combines an item from a positive pair with a random catalog item. The
implementations cap the quadratic pair blowup for very active users and work
in index space throughout — see `two_tower.create_pairs_for_user` and
`two_tower.create_negative_pairs`.

In [ ]:
positive_pairs = create_positive_pairs(train_items_idx, max_pairs_per_user=50)
negative_pairs = create_negative_pairs(positive_pairs, num_items,
                                       num_neg_per_pos=5)
print(f"{len(positive_pairs):,} positive pairs, "
      f"{len(negative_pairs):,} negative pairs")

### Spot check

Inspecting a sample of pairs before training catches data bugs early. We
follow one anchor movie through the whole chapter — Toy Story if it is in the
sample, otherwise the most-rated movie.

In [ ]:
def spot_check_movie() -> int:
    for idx, title in idx_to_title.items():
        if title.startswith("Toy Story (1995)"):
            return idx
    return int(train_df.item_idx.value_counts().idxmax())

anchor_idx = spot_check_movie()
print(f"--- {idx_to_title[anchor_idx]} ---")
shown = 0
for a, b in positive_pairs:
    if anchor_idx in (a, b):
        print(f"  p {idx_to_title[b if a == anchor_idx else a]}")
        shown += 1
        if shown == 3:
            break
for a, b in negative_pairs:
    if a == anchor_idx:
        print(f"  n {idx_to_title[b]}")
        shown += 1
        if shown == 6:
            break

## 3. The baseline: two towers, BCE, random negatives (Listing 5.3)

`TwoTower` in the package returns **logits** from `forward` because its loss
is `BCEWithLogitsLoss`, which applies the sigmoid internally — applying a
sigmoid in `forward` as well would squash every prediction into roughly
(0.5, 0.73) and cripple the gradients. Use `model.predict()` when you want
probabilities at inference time.

In [ ]:
BCE_PARAMS = dict(emb_dim=128, epochs=10, batch_size=2048, lr=1e-3)

with mlflow.start_run(run_name="bce-random-negatives") as bce_run:
    mlflow.log_params(BCE_PARAMS)
    bce_model, bce_losses = train_bce(
        TwoTower(num_items, emb_dim=BCE_PARAMS["emb_dim"]),
        positive_pairs, negative_pairs,
        epochs=BCE_PARAMS["epochs"],
        batch_size=BCE_PARAMS["batch_size"],
        lr=BCE_PARAMS["lr"],
        device=DEVICE,
    )
    for step, loss in enumerate(bce_losses):
        mlflow.log_metric("train_loss", loss, step=step)

bce_query, bce_cand = extract_embeddings(bce_model, device=DEVICE)

In [ ]:
def show_neighbors(query_vecs, cand_vecs, idx, k=5):
    scores = cand_vecs @ query_vecs[idx]
    scores[idx] = -np.inf
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.3f}  {idx_to_title[int(i)]}")

print(f"Neighbors of {idx_to_title[anchor_idx]} (BCE + random negatives):")
show_neighbors(bce_query, bce_cand, anchor_idx)

## 4. InfoNCE with hard negative mining (Sections 5.3.2–5.3.3)

The architecture does not change — `TwoTowerWithInfoNCE` subclasses
`TwoTower` and swaps only the objective and the negative sampling. Two
details of the package implementation matter enough to call out:

1. **Hard negatives come from a band of ranks (30–150), not the strict
   top-k.** The very top of the ranking is where false negatives hide —
   items the user would have loved but happened not to interact with (the
   Section 5.1.1 risk). Mine the strict top-k and you punish the model for
   its best predictions; the loss plateaus at chance level (ln(9) ≈ 2.20 for
   one positive against eight negatives — a number worth recognizing on
   sight) and the space collapses toward uniformity.
2. **Training warms up on random negatives** for two epochs, so mining
   operates on an embedding space that already has broad structure.

In [ ]:
INFONCE_PARAMS = dict(emb_dim=128, num_hard=4, num_rand=6, temperature=0.07,
                      epochs=20, warmup_epochs=2, batch_size=1024, lr=5e-3)

with mlflow.start_run(run_name="infonce-hard-negatives") as infonce_run:
    mlflow.log_params(INFONCE_PARAMS)
    infonce_model, infonce_losses = train_infonce(
        TwoTowerWithInfoNCE(
            num_items,
            emb_dim=INFONCE_PARAMS["emb_dim"],
            temperature=INFONCE_PARAMS["temperature"],
            num_hard=INFONCE_PARAMS["num_hard"],
            num_rand=INFONCE_PARAMS["num_rand"],
        ),
        positive_pairs,
        epochs=INFONCE_PARAMS["epochs"],
        warmup_epochs=INFONCE_PARAMS["warmup_epochs"],
        batch_size=INFONCE_PARAMS["batch_size"],
        lr=INFONCE_PARAMS["lr"],
        device=DEVICE,
    )
    for step, loss in enumerate(infonce_losses):
        mlflow.log_metric("train_loss", loss, step=step)

infonce_query, infonce_cand = extract_embeddings(infonce_model, device=DEVICE)

print(f"\nNeighbors of {idx_to_title[anchor_idx]} (InfoNCE + hard negatives):")
show_neighbors(infonce_query, infonce_cand, anchor_idx)

## 5. What difference does it make? (Section 5.3.4)

Evaluation follows the Chapter 4 protocol via
`evaluate_embedding_space`: each test user's retrieval is seeded from their
most recent **training** item, their training history is excluded from the
candidates, and the retrieved list is compared against their relevance set.
Exact search is used here so ANN approximation error cannot muddy the model
comparison — the ANN index enters in the next notebook.

In [ ]:
results = pd.DataFrame({
    "BCE + random negatives": evaluate_embedding_space(
        bce_query, bce_cand, relevance_sets, train_items_idx),
    "InfoNCE + hard negatives": evaluate_embedding_space(
        infonce_query, infonce_cand, relevance_sets, train_items_idx),
}).T

metric_map = {
    "Recall@100": "recall_at_100",
    "Precision@10": "precision_at_10",
    "NDCG@10": "ndcg_at_10",
}
for run, row_name in [
    (bce_run, "BCE + random negatives"),
    (infonce_run, "InfoNCE + hard negatives"),
]:
    with mlflow.start_run(run_id=run.info.run_id):
        mlflow.log_metrics({v: results.loc[row_name, k] for k, v in metric_map.items()})

results

The absolute numbers are modest by design — most test users have already
rated many movies. What matters is the **relative** improvement: InfoNCE with
hard negatives recovers substantially more relevant items at the retrieval
stage, and that improvement propagates through every downstream stage,
because a missed item here can never be recovered.

## 6. Saving artifacts for the other notebooks

In [ ]:
ART = project_root / "data" / "processed" / "chapter05"
ART.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    ART / "embeddings.npz",
    bce_query=bce_query, bce_cand=bce_cand,
    infonce_query=infonce_query, infonce_cand=infonce_cand,
)
with open(ART / "mappings.json", "w") as fh:
    json.dump({
        "item_ids": item_ids,
        "idx_to_title": {str(k): v for k, v in idx_to_title.items()},
        "idx_to_genres": {str(k): v for k, v in idx_to_genres.items()},
        "anchor_idx": anchor_idx,
    }, fh)
cols = ["userId", "movieId", "item_idx", "rating", "timestamp"]
train_df[cols].to_csv(ART / "train.csv", index=False)
test_df[cols].to_csv(ART / "test.csv", index=False)
print("Saved:", sorted(p.name for p in ART.iterdir()))

## Summary

The two-tower model earns its place at the retrieval stage through one
property: the towers encode independently, so candidate embeddings can be
precomputed. The quality of that embedding space is the ceiling on retrieval
quality, and two training changes raise the ceiling considerably — hard
negatives make the training problem realistic, and InfoNCE strengthens the
signal by comparing the positive against many negatives at once. The
architecture never changed; only the objective did. In the next notebook we
make retrieval over this space fast with an Approximate Nearest Neighbor
index.